[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/solutions/part_05_classical_ml/ch24_trees_ensembles_interpretability/02_guided_todos_SOLUTION.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `ANNOTATED SOLUTION NOTEBOOK`


# Chapter 24 Guided To-Do Practice Labs

**Book Reference:** Chapter 24, Section 24.5 (Guided To-Do Practice)  
Run **Step 0** to activate `sharukh-tutor`, implement all 6 tree, ensemble, and interpretability functions, and pass each **function name (without parentheses!)** to `tutor.check(...)`. Use `tutor.hint("ch24_todo_XX", level=1..3)` whenever you want a progressive nudge.

In [ ]:
# @title Step 0: Click Play (▶) to Activate Interactive Hints & Auto-Checker
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import numpy as np
import pandas as pd
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.tree import DecisionTreeClassifier
import sharukh_tutor as tutor

tutor.activate(chapter="ch24", notebook="guided_todos")


---
## To-Do 24.1 (Impurity Criterion): `gini_impurity(y)`

Implement `gini_impurity(y)` in pure NumPy to compute $G(y) = 1 - \sum_k p_k^2$:
- Return `0.0` if `y` is empty (`arr.size == 0`).
- Otherwise compute class proportions via `np.unique(arr, return_counts=True)` and return `round(float(1.0 - np.sum(probs ** 2)), 4)`.

In [ ]:
def gini_impurity(y):
    """Compute Gini impurity 1 - sum(p_k^2) rounded to 4 decimals."""
    arr = np.asarray(y)
    if arr.size == 0:
        return 0.0
    _, counts = np.unique(arr, return_counts=True)
    probs = counts / arr.size
    return round(float(1.0 - np.sum(probs ** 2)), 4)


In [ ]:
tutor.check("ch24_todo_01", gini_impurity)


---
## To-Do 24.2 (Information Theory): `shannon_entropy(y)`

Implement `shannon_entropy(y)` to compute base-2 entropy $H(y) = -\sum_k p_k \log_2(p_k)$:
- Return `0.0` for empty or pure arrays (avoiding `-0.0`).
- Round the returned Python `float` to **4 decimal places**.

In [ ]:
def shannon_entropy(y):
    """Compute base-2 Shannon entropy rounded to 4 decimals."""
    arr = np.asarray(y)
    if arr.size == 0:
        return 0.0
    _, counts = np.unique(arr, return_counts=True)
    probs = counts / arr.size
    probs = probs[probs > 0]
    ent = float(-np.sum(probs * np.log2(probs)))
    return round(0.0 if ent == 0.0 else ent, 4)


In [ ]:
tutor.check("ch24_todo_02", shannon_entropy)


---
## To-Do 24.3 (Greedy CART Splitting): `best_stump_split(x_feature, y)`

Implement `best_stump_split(x_feature, y)`:
- Find all midpoints between consecutive sorted unique values of `x_feature`.
- For each threshold $t$, split `y` into `left = y[x <= t]` and `right = y[x > t]` and compute weighted Gini impurity $\frac{n_L}{n} G(y_L) + \frac{n_R}{n} G(y_R)$.
- Return `(best_thresh, best_weighted_gini)` as a 2-tuple of floats rounded to **4 decimal places**.

In [ ]:
def best_stump_split(x_feature, y):
    """Find midpoint threshold minimizing weighted Gini impurity."""
    def _gini(arr):
        if arr.size == 0:
            return 0.0
        _, counts = np.unique(arr, return_counts=True)
        probs = counts / arr.size
        return float(1.0 - np.sum(probs ** 2))

    x = np.asarray(x_feature, dtype=float)
    y_arr = np.asarray(y)
    uniq = np.unique(x)
    mids = (uniq[:-1] + uniq[1:]) / 2.0
    best_t, best_g = float(mids[0]), float("inf")
    n = y_arr.size
    for t in mids:
        left = y_arr[x <= t]
        right = y_arr[x > t]
        wg = (left.size / n) * _gini(left) + (right.size / n) * _gini(right)
        if wg < best_g:
            best_g = wg
            best_t = float(t)
    return (round(best_t, 4), round(best_g, 4))


In [ ]:
tutor.check("ch24_todo_03", best_stump_split)


---
## To-Do 24.4 (Tree Overfitting Audit): `compare_tree_depths(X_train, y_train, X_val, y_val, depths, seed=42)`

Implement `compare_tree_depths(...)`:
- For each `d` in `depths`, fit `DecisionTreeClassifier(max_depth=d, random_state=seed)` on `(X_train, y_train)`.
- Compute `train_auc` and `val_auc` using `roc_auc_score` on positive-class probabilities (`predict_proba(...)[:, 1]`), plus `gap = train_auc - val_auc`.
- Return a `pd.DataFrame` with columns `["max_depth", "train_auc", "val_auc", "gap"]` rounded to **4 decimal places**.

In [ ]:
def compare_tree_depths(X_train, y_train, X_val, y_val, depths, seed=42):
    """Compare DecisionTree train/val ROC-AUC and gap across depths."""
    rows = []
    for d in depths:
        clf = DecisionTreeClassifier(max_depth=d, random_state=seed)
        clf.fit(X_train, y_train)
        tr = float(roc_auc_score(y_train, clf.predict_proba(X_train)[:, 1]))
        va = float(roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1]))
        rows.append({
            "max_depth": d,
            "train_auc": round(tr, 4),
            "val_auc": round(va, 4),
            "gap": round(tr - va, 4),
        })
    return pd.DataFrame(rows)


In [ ]:
tutor.check("ch24_todo_04", compare_tree_depths)


---
## To-Do 24.5 (Ensemble Benchmark): `fit_ensemble_comparison(X_train, y_train, X_test, y_test, seed=42)`

Implement `fit_ensemble_comparison(...)` fitting three models with `random_state=seed`:
- `"DecisionTree"`: `DecisionTreeClassifier(max_depth=5, random_state=seed)`
- `"RandomForest"`: `RandomForestClassifier(n_estimators=100, max_depth=6, random_state=seed)`
- `"HistGradientBoosting"`: `HistGradientBoostingClassifier(max_depth=4, random_state=seed)`
- Return a `pd.DataFrame` with columns `["model", "roc_auc", "pr_auc"]` rounded to **4 decimal places**.

In [ ]:
def fit_ensemble_comparison(X_train, y_train, X_test, y_test, seed=42):
    """Fit DecisionTree, RandomForest, and HistGradientBoosting."""
    models = [
        ("DecisionTree", DecisionTreeClassifier(
            max_depth=5, random_state=seed)),
        ("RandomForest", RandomForestClassifier(
            n_estimators=100, max_depth=6, random_state=seed)),
        ("HistGradientBoosting", HistGradientBoostingClassifier(
            max_depth=4, random_state=seed)),
    ]
    rows = []
    for name, clf in models:
        clf.fit(X_train, y_train)
        proba = clf.predict_proba(X_test)[:, 1]
        rows.append({
            "model": name,
            "roc_auc": round(float(roc_auc_score(y_test, proba)), 4),
            "pr_auc": round(
                float(average_precision_score(y_test, proba)), 4
            ),
        })
    return pd.DataFrame(rows)


In [ ]:
tutor.check("ch24_todo_05", fit_ensemble_comparison)


---
## To-Do 24.6 (Held-Out Interpretability): `compute_permutation_ranking(model, X_test, y_test, feature_names, seed=42)`

Implement `compute_permutation_ranking(...)`:
- Run `permutation_importance(model, X_test, y_test, scoring="roc_auc", n_repeats=10, random_state=seed)`.
- Return a `pd.DataFrame` with columns `["feature", "importance_mean", "importance_std"]` sorted by `importance_mean` **descending** and rounded to **4 decimal places**.

In [ ]:
def compute_permutation_ranking(
    model, X_test, y_test, feature_names, seed=42
):
    """Rank features by held-out ROC-AUC permutation importance."""
    res = permutation_importance(
        model, X_test, y_test, scoring="roc_auc",
        n_repeats=10, random_state=seed,
    )
    df = pd.DataFrame({
        "feature": list(feature_names),
        "importance_mean": np.round(res.importances_mean, 4),
        "importance_std": np.round(res.importances_std, 4),
    })
    return df.sort_values(
        "importance_mean", ascending=False
    ).reset_index(drop=True)


In [ ]:
tutor.check("ch24_todo_06", compute_permutation_ranking)


---
## Annotated Debugging Post-Mortem (Solution Notebook Only)

- **To-Do 24.2 (Base-2 vs. Natural Log):** Information-theoretic entropy uses `np.log2` (measured in bits, so a 50/50 split equals exactly `1.0`). Using `np.log` produces `0.6931` nats and fails the bit-scale check.
- **To-Do 24.4 (`predict_proba` vs. `.predict` for ROC-AUC):** Passing hard class predictions (`clf.predict(X)`) into `roc_auc_score` throws away ranking granularity across decision thresholds. Always pass positive-class probabilities `clf.predict_proba(X)[:, 1]`.
- **To-Do 24.6 (Held-Out vs. Training Permutation Importance):** Running `permutation_importance` on `X_train` measures memorization rather than generalization! Evaluating on `X_test` reveals which features genuinely preserve out-of-sample discrimination.

In [ ]:
tutor.summary()
